# Cosmos Geocoding RP Stats

Runs Cosmos DB queries against the APT collection to generate Geocoding routing-point (RP) stats for a given country and SN change window.

**Metrics produced** (distinct `sourceAptId` counts, scoped by `countryIso`, `status = SEEDED`, `_ts` window, and transformation users):

1. **Total impacted APTs** — overall change footprint.
2. **Geocoding RP regression** — APTs with an `ACCESS / DELETE` of `GEOCODING` and no matching `ADD` (potential coverage loss).
3. **Coverage improved** — APTs with an `ACCESS / ADD` of `GEOCODING` and no matching `DELETE` (newly enabled geocoding).

Reference: [Useful Cosmos Queries for stats](https://tomtom.atlassian.net/wiki/spaces/POPSPUNE/pages/1779206225/Useful+Cosmos+Queries+for+stats).

## 1. Install dependencies

In [17]:
# Install the Cosmos SDK on the cluster, then restart Python so the new package is importable.
%pip install --quiet azure-cosmos

Note: you may need to restart the kernel to use updated packages.


## 2. Connect to Cosmos DB

In [ ]:
# Connect to Azure Cosmos DB
from azure.cosmos import CosmosClient

# Pull credentials from Databricks secrets (set scope/keys once via the Databricks CLI / UI)
COSMOS_SCOPE   = "orbis-addressing"
COSMOS_ENDPOINT = "https://opas-merone-prod-db.documents.azure.com:443/" #dbutils.secrets.get(scope=COSMOS_SCOPE, key="cosmos-endpoint")
COSMOS_KEY      = "" #dbutils.secrets.get(scope=COSMOS_SCOPE, key="cosmos-key")

DATABASE_NAME  = "opas-merone-prod-db"
CONTAINER_NAME = "address-point-message"

client    = CosmosClient(COSMOS_ENDPOINT, credential=COSMOS_KEY)
database  = client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

print(f"Connected to {DATABASE_NAME}.{CONTAINER_NAME}")

Connected to opas-merone-prod-db.address-point-message


## 3. Parameters

Shared by all three queries below. Update for the country and SN change window you want to analyse — `_ts` values are Unix epoch seconds (UTC).

In [19]:
countryIso           = "SVK"
startTimeStampEpoch  = 1778176800
endTimestampEpoch    = 1778209200

QUERY_PARAMS = [
    {"name": "@countryIso", "value": countryIso},
    {"name": "@startTs",    "value": startTimeStampEpoch},
    {"name": "@endTs",      "value": endTimestampEpoch},
]

def run_count_query(label: str, query: str) -> int:
    result = list(container.query_items(
        query=query,
        parameters=QUERY_PARAMS,
        enable_cross_partition_query=True,
    ))
    count = result[0] if result else 0
    print(f"{label}: {count}")
    return count

## 4. Query 1 — Total impacted APTs

Distinct count of `sourceAptId`s touched by the SN change window. Gauges overall impact size.

In [20]:
TOTAL_IMPACTED_APTS_QUERY = """
SELECT VALUE COUNT(1)
FROM (
    SELECT DISTINCT c.rawSourceApt.sourceAptId
    FROM c
    WHERE c.rawSourceApt.countryIso = @countryIso
      AND c.status = "SEEDED"
      AND c._ts > @startTs AND c._ts < @endTs
      AND (c.user = "TRANSFORMATION_SUBSCRIPTION" OR c.user = "TRANSFORMATION_PARENT")
)
"""

total_impacted_apts = run_count_query("Total impacted APTs", TOTAL_IMPACTED_APTS_QUERY)

Total impacted APTs: 0


## 5. Query 2 — Geocoding RP regression (DELETE without corresponding ADD)

APTs whose Geocoding access was removed without a matching add — flags potential coverage loss.

In [21]:
GEOCODING_REGRESSION_QUERY = """
SELECT VALUE COUNT(1)
FROM (
    SELECT DISTINCT c.rawSourceApt.sourceAptId
    FROM c
    WHERE EXISTS (
        SELECT VALUE d
        FROM d IN c.deltas
        WHERE d.type = 'ACCESS'
          AND d.changeType = 'DELETE'
          AND ARRAY_CONTAINS(d.accessChange.types, 'GEOCODING')
    )
    AND NOT EXISTS (
        SELECT VALUE d
        FROM d IN c.deltas
        WHERE d.type = 'ACCESS'
          AND d.changeType = 'ADD'
          AND ARRAY_CONTAINS(d.accessChange.types, 'GEOCODING')
    )
    AND c.rawSourceApt.countryIso = @countryIso
    AND c.status = "SEEDED"
    AND c._ts > @startTs AND c._ts < @endTs
    AND (c.user = "TRANSFORMATION_SUBSCRIPTION" OR c.user = "TRANSFORMATION_PARENT")
)
"""

geocoding_regression = run_count_query("Geocoding RP regression (DELETE without ADD)", GEOCODING_REGRESSION_QUERY)

Geocoding RP regression (DELETE without ADD): 0


## 6. Query 3 — Coverage improved (ADD without corresponding DELETE)

APTs that gained Geocoding access without a matching delete — newly enabled geocoding coverage.

In [22]:
COVERAGE_IMPROVED_QUERY = """
SELECT VALUE COUNT(1)
FROM (
    SELECT DISTINCT c.rawSourceApt.sourceAptId
    FROM c
    WHERE EXISTS (
        SELECT VALUE d
        FROM d IN c.deltas
        WHERE d.type = 'ACCESS'
          AND d.changeType = 'ADD'
          AND ARRAY_CONTAINS(d.accessChange.types, 'GEOCODING')
    )
    AND NOT EXISTS (
        SELECT VALUE d
        FROM d IN c.deltas
        WHERE d.type = 'ACCESS'
          AND d.changeType = 'DELETE'
          AND ARRAY_CONTAINS(d.accessChange.types, 'GEOCODING')
    )
    AND c.rawSourceApt.countryIso = @countryIso
    AND c.status = "SEEDED"
    AND c._ts > @startTs AND c._ts < @endTs
    AND (c.user = "TRANSFORMATION_SUBSCRIPTION" OR c.user = "TRANSFORMATION_PARENT")
)
"""

coverage_improved = run_count_query("Coverage improved (ADD without DELETE)", COVERAGE_IMPROVED_QUERY)

Coverage improved (ADD without DELETE): 0
